# LLM Simulation Workbench

This notebook runs the LLM rewriting simulation pipeline: it loads datasets, applies personality/style rewrites, tests multiple providers, and exports outputs for factual consistency auditing.

## 1) Initial Setup

In this step, we import core libraries and load environment variables to validate API keys.

In [ ]:
import json
import os
import re
import time
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

In [ ]:
load_dotenv()

for env_var in ["GEMINI_API_KEY", "OPENROUTER_API_KEY", "DEEPSEEK_API_KEY"]:
    if os.getenv(env_var):
        print(f"{env_var} loaded from .env successfully.")
    else:
        print(f"{env_var} not found. Check the .env file.")

## 2) Free Model Catalog (OpenRouter)

Here we query OpenRouter for available models and list only the ones that end with `:free`.

In [ ]:
api_key = os.getenv("OPENROUTER_API_KEY")
if api_key:
    headers = {"Authorization": f"Bearer {api_key}"}

    resp = requests.get("https://openrouter.ai/api/v1/models", headers=headers, timeout=30)
    resp.raise_for_status()

    models = resp.json().get("data", [])
    free_models = sorted([m["id"] for m in models if ":free" in m.get("id", "")])

    print(f"Total active free models: {len(free_models)}")
    for mid in free_models[:30]:
        print(mid)

## 3) Dataset Loading and Organization

In this section, we load files from the `data` folder, build DataFrames, and map names for easier experimentation.

In [ ]:
def read_dataset(file_path: Path) -> pd.DataFrame:
    suffix = file_path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(file_path)
    if suffix == ".json":
        return pd.read_json(file_path)

    raise ValueError(f"Unsupported file format: {file_path.name}")


data_dir = Path("../data")
dataset_files = sorted(
    [*data_dir.glob("*.csv"), *data_dir.glob("*.json")],
    key=lambda path: path.name.lower(),
)

if not dataset_files:
    raise FileNotFoundError("No CSV or JSON file was found in the 'data' folder.")

dataset_files

In [ ]:
def to_variable_name(file_path: Path) -> str:
    name = file_path.stem.lower()
    name = re.sub(r"\W+", "_", name)
    name = re.sub(r"^(\d)", r"df_\1", name)
    return name


dataframes = {}

for dataset_file in dataset_files:
    df_name = to_variable_name(dataset_file)
    df = read_dataset(dataset_file)
    dataframes[df_name] = df

list(dataframes.keys())

In [ ]:
for df_name, df in dataframes.items():
    print(f"{df_name}: {df.shape[0]} rows x {df.shape[1]} columns")

## 4) Simulation and Rewriting Utilities

In this section, we import utility functions and the provider enum to standardize calls (`Provider.GEMINI`, `Provider.OPENROUTER`, `Provider.LOCAL`).

In [ ]:
import utils.simulation_functions as simf
from consts.llm_requests import PROMPT_TEMPLATE
from enums import DefaultPersonality, Models, Provider
from utils.run_report import append_execution_report, resolve_execution_report_path
from utils.simulation_functions import rewrite_news_with_personality

In [ ]:
DATASET = "newsdata_news"
PERSONALITY = DefaultPersonality.ConservativeRight

TEST_MAX_ROWS = 50
RETRY_ATTEMPTS = 2
TEXT_COLUMN = "description"

QUERY_METADATA_ROW_ID = "__query_metadata__"
NEWSDATA_QUERY_METADATA = None
source_df = dataframes[DATASET].copy()

if "article_id" in source_df.columns:
    metadata_mask = (
        source_df["article_id"].astype(str).str.strip().str.lower().eq(QUERY_METADATA_ROW_ID)
    )

    if metadata_mask.any():
        metadata_row = source_df.loc[metadata_mask].iloc[0]
        raw_metadata = str(metadata_row.get("description", "")).strip()

        if raw_metadata:
            try:
                NEWSDATA_QUERY_METADATA = json.loads(raw_metadata)
            except json.JSONDecodeError:
                NEWSDATA_QUERY_METADATA = {"raw_description": raw_metadata}
        else:
            NEWSDATA_QUERY_METADATA = {}

        source_df = source_df.loc[~metadata_mask].copy()

# Global defaults for subsequent rewrite_news_with_personality calls.
simf.DEFAULT_DF = source_df
simf.DEFAULT_PERSONALITY = PERSONALITY
simf.DEFAULT_MAX_ROWS = TEST_MAX_ROWS
simf.DEFAULT_RETRY_ATTEMPTS = RETRY_ATTEMPTS
simf.DEFAULT_TEXT_COLUMN = TEXT_COLUMN

rewrite_run_metrics = []

## 5) Simulation with Remote APIs

Next, we run rewriting simulations with remote providers and inspect `rewrite_status` and `rewrite_error`.

In [ ]:
_t0 = time.perf_counter()
gemini_rewritten_df = rewrite_news_with_personality(
    provider=Provider.GEMINI,
    model=Models.GEMINI31FlashLite,
    max_requests_per_minute=15,
)
gemini_elapsed_seconds = time.perf_counter() - _t0

print("Gemini status:")
print(gemini_rewritten_df["rewrite_status"].value_counts(dropna=False))

errors = gemini_rewritten_df[gemini_rewritten_df["rewrite_status"] == "error"]
if not errors.empty:
    print("\nErrors in Gemini:")
    for idx, row in errors.iterrows():
        print(f"- row {idx}: {row['rewrite_error']}")

rewrite_run_metrics.append(
    {
        "dataset": DATASET,
        "output_name": "gemini_rewritten_df",
        "provider": Provider.GEMINI.value,
        "model": str(Models.GEMINI31FlashLite),
        "duration_seconds": round(gemini_elapsed_seconds, 3),
        "rows_requested": min(TEST_MAX_ROWS, len(dataframes[DATASET])),
        "rows_success": int((gemini_rewritten_df["rewrite_status"] == "success").sum()),
        "rows_error": int((gemini_rewritten_df["rewrite_status"] == "error").sum()),
    }
)

In [ ]:
_t0 = time.perf_counter()
openrouter_nvidia_rewritten_df = rewrite_news_with_personality(
    provider=Provider.OPENROUTER,
    model=Models.NEMOTRON3Super120B,
    max_requests_per_minute=20,
)
openrouter_nvidia_elapsed_seconds = time.perf_counter() - _t0

print("OpenRouter (Nvidia Nemotron 3 Super free) status:")
print(openrouter_nvidia_rewritten_df["rewrite_status"].value_counts(dropna=False))

errors = openrouter_nvidia_rewritten_df[openrouter_nvidia_rewritten_df["rewrite_status"] == "error"]
if not errors.empty:
    print("\nErrors in OpenRouter Nvidia:")
    for idx, row in errors.iterrows():
        print(f"- row {idx}: {row['rewrite_error']}")

rewrite_run_metrics.append(
    {
        "dataset": DATASET,
        "output_name": "openrouter_nvidia_rewritten_df",
        "provider": Provider.OPENROUTER.value,
        "model": str(Models.NEMOTRON3Super120B),
        "duration_seconds": round(openrouter_nvidia_elapsed_seconds, 3),
        "rows_requested": min(TEST_MAX_ROWS, len(dataframes[DATASET])),
        "rows_success": int((openrouter_nvidia_rewritten_df["rewrite_status"] == "success").sum()),
        "rows_error": int((openrouter_nvidia_rewritten_df["rewrite_status"] == "error").sum()),
    }
)

In [ ]:
_t0 = time.perf_counter()
openrouter_step_rewritten_df = rewrite_news_with_personality(
    provider=Provider.OPENROUTER,
    model=Models.STEP35FLASH,
    max_requests_per_minute=20,
)
openrouter_step_elapsed_seconds = time.perf_counter() - _t0

print("OpenRouter (Step 3.5 Flash) status:")
print(openrouter_step_rewritten_df["rewrite_status"].value_counts(dropna=False))

errors = openrouter_step_rewritten_df[openrouter_step_rewritten_df["rewrite_status"] == "error"]
if not errors.empty:
    print("\nErrors in OpenRouter Step 3.5 Flash:")
    for idx, row in errors.iterrows():
        print(f"- row {idx}: {row['rewrite_error']}")

rewrite_run_metrics.append(
    {
        "dataset": DATASET,
        "output_name": "openrouter_step_rewritten_df",
        "provider": Provider.OPENROUTER.value,
        "model": str(Models.STEP35FLASH),
        "duration_seconds": round(openrouter_step_elapsed_seconds, 3),
        "rows_requested": min(TEST_MAX_ROWS, len(dataframes[DATASET])),
        "rows_success": int((openrouter_step_rewritten_df["rewrite_status"] == "success").sum()),
        "rows_error": int((openrouter_step_rewritten_df["rewrite_status"] == "error").sum()),
    }
)

## 6) Local Model Test (Ollama)

In this section, we call a local OpenAI-compatible endpoint (`http://127.0.0.1:11434/v1`) to validate generation without external APIs.

In [ ]:
_t0 = time.perf_counter()
local_llama_rewritten_df = rewrite_news_with_personality(
    provider=Provider.LOCAL,
    model=Models.LLAMA318B,
    base_url="http://127.0.0.1:11434/v1",
    api_key="ollama",
)

local_llama_elapsed_seconds = time.perf_counter() - _t0

print("Local Llama status:")
print(local_llama_rewritten_df["rewrite_status"].value_counts(dropna=False))

local_llama_rewritten_df[
    [
        "title",
        "source_text_column",
        "target_language",
        "target_language_source",
        "rewritten_news",
        "rewrite_status",
        "rewrite_error",
    ]
].head()

rewrite_run_metrics.append(
    {
        "dataset": DATASET,
        "output_name": "local_llama_rewritten_df",
        "provider": Provider.LOCAL.value,
        "model": str(Models.LLAMA318B),
        "duration_seconds": round(local_llama_elapsed_seconds, 3),
        "rows_requested": min(TEST_MAX_ROWS, len(dataframes[DATASET])),
        "rows_success": int((local_llama_rewritten_df["rewrite_status"] == "success").sum()),
        "rows_error": int((local_llama_rewritten_df["rewrite_status"] == "error").sum()),
    }
)

## 7) Export Simulation Outputs

In this step, we export final rewritten DataFrames to `../output/runs/<run_id>/rewritten` for use in the audit notebooks.

In [ ]:
run_id_env = os.getenv("RUN_ID")
if run_id_env:
    run_id = run_id_env
    output_dir = Path("../output/runs") / run_id / "rewritten"
else:
    run_id = None
    output_dir = Path("../output/rewritten")

output_dir.mkdir(parents=True, exist_ok=True)
if run_id:
    print(f"RUN_ID: {run_id}")
else:
    print("RUN_ID not set. Using default folder ../output/rewritten.")
print(f"Exporting rewrites to: {output_dir}")


def _safe_output_name(name: str) -> str:
    cleaned = re.sub(r"\W+", "_", name).strip("_").lower()
    return cleaned or "rewritten_dataset"


candidate_dfs: dict[str, pd.DataFrame] = {}
global_items = list(globals().items())

for var_name, var_value in global_items:
    if not isinstance(var_value, pd.DataFrame):
        continue

    # Export only final rewrite results, e.g.: gemini_rewritten_df
    if not var_name.endswith("_rewritten_df"):
        continue

    required_columns = {"rewritten_news", "rewrite_status"}
    if not required_columns.issubset(set(var_value.columns)):
        continue

    candidate_dfs[var_name] = var_value

if not candidate_dfs:
    print("No final rewrite DataFrame was found for export.")
else:
    exported_files = []
    skipped_all_error = []

    for df_name in sorted(candidate_dfs.keys()):
        df_value = candidate_dfs[df_name]

        # Keep only successful rows (no error and with rewritten text).
        export_df = df_value[
            (df_value["rewrite_status"] == "success")
            & df_value["rewritten_news"].notna()
            & (df_value["rewritten_news"].astype(str).str.strip() != "")
        ].copy()

        if export_df.empty:
            skipped_all_error.append(df_name)
            continue

        output_name = f"{_safe_output_name(df_name)}.csv"
        output_path = output_dir / output_name
        export_df.to_csv(output_path, index=False, encoding="utf-8-sig")
        exported_files.append(output_path)

    print(f"Total exported DataFrames: {len(exported_files)}")
    for file_path in exported_files:
        print(f"- {file_path}")

    if skipped_all_error:
        print("\nDataFrames not exported (no valid rows):")
        for df_name in skipped_all_error:
            print(f"- {df_name}")

run_id_for_report, report_path = resolve_execution_report_path()

source_df = (
    simf.DEFAULT_DF.copy()
    if isinstance(simf.DEFAULT_DF, pd.DataFrame)
    else dataframes.get(DATASET, pd.DataFrame()).copy()
)
query_details = {}
for col in ["language", "country", "category", "keywords", "source_name"]:
    if col in source_df.columns:
        values = source_df[col].dropna().astype(str).str.strip()
        values = values[values != ""]
        if not values.empty:
            query_details[col] = sorted(values.unique().tolist())[:10]

if not query_details:
    query_details = "not available in selected dataset columns"

llm_models_used = [f"{item['provider']}::{item['model']}" for item in rewrite_run_metrics]

dataset_report_details = {
    "dataset_selected": DATASET,
    "dataset_total_rows_loaded": int(len(source_df)),
    "dataset_rows_planned_per_model": int(min(TEST_MAX_ROWS, len(source_df))),
    "dataset_column_count": int(source_df.shape[1]),
    "dataset_columns": source_df.columns.astype(str).tolist(),
    "dataset_value_summary": query_details,
}

if isinstance(NEWSDATA_QUERY_METADATA, dict) and NEWSDATA_QUERY_METADATA:
    dataset_report_details["newsdata_query_details"] = NEWSDATA_QUERY_METADATA

append_execution_report(
    report_path=report_path,
    notebook_name="llm_simulation_workbench.ipynb",
    section_title="Dataset query and profile",
    run_id=run_id_for_report,
    details=dataset_report_details,
)

llm_execution_details = {
    "llm_models_used": llm_models_used,
    "personality_used": PERSONALITY,
    "prompt_template_used": PROMPT_TEMPLATE,
    "rewrite_metrics": rewrite_run_metrics,
    "total_rewritten_success": int(sum(item["rows_success"] for item in rewrite_run_metrics)),
}

append_execution_report(
    report_path=report_path,
    notebook_name="llm_simulation_workbench.ipynb",
    section_title="LLM rewriting execution",
    run_id=run_id_for_report,
    details=llm_execution_details,
)

print(f"Execution report updated: {report_path}")